In [67]:
from dataclasses import dataclass
from torch.utils.data import Dataset
from datasets import load_dataset
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import time
import os
import matplotlib as plt

In [68]:
class TextDataset(Dataset):
  def __init__(self, texts:list[str], tokenizer, max_seq_len: int = 2048):
    self.tokenizer = tokenizer
    self.max_seq_len = max_seq_len

    full_tokens = []

    for text in texts:
      tokens = tokenizer.encode(text)
      full_tokens.extend(tokens)
      full_tokens.append(tokenizer.eos_token_id)

    self.full_tokens = torch.tensor(full_tokens, dtype=torch.long)
    print(f"Total tokens: {len(self.full_tokens)}")

  def __len__(self):
    return(len(self.full_tokens) - 1) // self.max_seq_len


  def __getitem__(self, idx: int) -> tuple:
    start = idx * self.max_seq_len
    end = start + self.max_seq_len

    input_ids = self.full_tokens[start:end]
    target_ids = self.full_tokens[start + 1: end + 1]

    return input_ids, target_ids

@dataclass
class TokenizerConfig:
  name: str = "gpt2"
  vocab_size: int = 50257

class Tokenizer:
  def __init__(self, config: TokenizerConfig = None):
    self.config = config or TokenizerConfig()
    self.enc = tiktoken.get_encoding(self.config.name)
    self.eos_token = "<|endoftext|>"
    self.eos_token_id = self.enc.encode(
        self.eos_token,
        allowed_special = {self.eos_token}
    )[0]

  def encode(self, text:str) -> list[int]:
    return self.enc.encode(text, allowed_special = {self.eos_token})

  def decode(self, ids: list[int]) -> str:
    return self.enc.decode(ids)

  def vocab_size(self) -> int:
    return self.config.vocab_size

class Embedding(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embedding_dim)
    self.embedding_dim = embedding_dim

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    embeddings = self.embed(x)
    scale_factor = 1.0
    embeddings_scaled = embeddings * scale_factor
    return embeddings_scaled


class RoPE(nn.Module):
  def __init__(self, embeddings_dim, max_seq_len: int = 2048, theta: float = 10000.):
    super().__init__()
    assert(embeddings_dim % 2 == 0)

    pair_number = torch.arange(0, (embeddings_dim // 2), 1).float()

    denom = theta ** (2*pair_number / embeddings_dim)
    inv_freq = 1. / denom

    positions = torch.arange(0, max_seq_len, 1)

    pair_angles = torch.outer(positions, inv_freq)

    angles = torch.repeat_interleave(pair_angles, repeats=2, dim=-1)

    sine = torch.sin(angles)
    cosine = torch.cos(angles)
    self.register_buffer("sine_cached", sine)
    self.register_buffer("cos_cached", cosine)

  @staticmethod
  def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rotated = torch.stack((-x_odd, x_even), dim=-1).flatten(start_dim=-2)
    return rotated

  def forward(self, x: torch.Tensor) -> torch.Tensor:

    seq_len = x.shape[-2]

    cos = self.cos_cached[:seq_len]
    sin = self.sine_cached[:seq_len]

    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)

    x_prime = x * cos + self.rotate_half(x) * sin
    return x_prime

class MultiHeadAttention(nn.Module):
  def __init__(self, embeddings_dim: int, head_count: int, dropout: float = .1):
    super().__init__()
    assert embeddings_dim % head_count == 0, "embeddings_dim must be divisible by head_count"
    dim_per_head = embeddings_dim // head_count

    self.head_count = head_count
    self.dim_per_head = dim_per_head
    self.dropout = dropout
    self.w_qkv = nn.Linear(embeddings_dim, 3*embeddings_dim, bias = False)
    self.w_o = nn.Linear(embeddings_dim, embeddings_dim, bias = False)
    self.rope = RoPE(dim_per_head)
    self.attn_dropout = nn.Dropout(dropout)
    self.resid_dropout = nn.Dropout(dropout)

  def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    batch_size, seq_len, embeddings_dim = x.shape
    qkv = self.w_qkv(x)
    qkv_reshape = qkv.reshape(batch_size, seq_len, 3, self.head_count, self.dim_per_head)
    qkv = qkv_reshape.permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

    q = self.rope(q)
    k = self.rope(k)

    scores = (q @k.transpose(-1, -2))
    scores_scaled = scores / math.sqrt(self.dim_per_head)
    if mask is not None:
      scores_scaled = scores_scaled.masked_fill(mask == 0, float('-inf'))
    softmax_weights = F.softmax(scores_scaled, dim=-1)
    softmax_weights_dropout = self.attn_dropout(softmax_weights)

    output = softmax_weights_dropout @ v

    output = output.transpose(1, 2).reshape(batch_size, seq_len, embeddings_dim)

    output = self.w_o(output)

    output = self.resid_dropout(output)

    return output

class RMSNorm(nn.Module):
    def __init__(self, embeddings_dim, eps=1e-5):

        super().__init__()

        # input: [batch, seq_len, embeddings_dim]
        self.gamma = nn.Parameter(torch.ones(embeddings_dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      # compute denom
      rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
      # multiply
      xrms = x / rms
      # gamma
      gxrms = self.gamma * xrms
      return gxrms

class SwiGLU(nn.Module):
    def __init__(self, embeddings_dim, expand: int = 4):

        super().__init__()

        self.W1 = nn.Linear(embeddings_dim, expand * embeddings_dim, bias=False)
        self.W2 = nn.Linear(embeddings_dim, expand * embeddings_dim, bias=False)
        self.W3 = nn.Linear(expand * embeddings_dim, embeddings_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # output = W3(SiLU(W1(x))*W2(x))
        # SiLU = x * sigmoid(x)
        expand = self.W1(x)
        gate = self.W2(x)

        # silu = lambda x : x * torch.sigmoid(x)
        return self.W3(F.silu(expand) * gate)

class TransformerBlock(nn.Module):

  def __init__(self, embeddings_dim, head_count, dropout: int = 0.1):
    super().__init__()
    self.rms1 = RMSNorm(embeddings_dim)
    self.rms2 = RMSNorm(embeddings_dim)
    self.attention = MultiHeadAttention(embeddings_dim, head_count, dropout)
    self.ffn = SwiGLU(embeddings_dim)

  def forward(self, x: torch.Tensor, mask : torch.Tensor = None) -> torch.Tensor:
    x = x + self.attention(self.rms1(x), mask)
    x = x + self.ffn(self.rms2(x))
    return x

class CosineWarmupScheduler:
  def __init__(self, optimizer, warmup_steps, max_steps, max_lr = 3e-4, min_lr = 1e-5):
    self.optimizer = optimizer
    self.warmup_steps = warmup_steps
    self.max_steps = max_steps
    self.max_lr = max_lr
    self.min_lr = min_lr
    self.current_step = 0

  def get_lr(self):
    if self.current_step < self.warmup_steps:
      return self.max_lr * self.current_step / self.warmup_steps
    elif self.current_step < self.max_steps:
      progress = (self.current_step - self.warmup_steps) / (self.max_steps - self.warmup_steps)
      cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
      return self.min_lr + (self.max_lr - self.min_lr) * cosine_decay
    return self.min_lr

  def step(self):
    lr = self.get_lr()
    for param_group in self.optimizer.param_groups:
      param_group["lr"] = lr
    self.current_step += 1

  def state_dict(self):
    return{"current_step": self.current_step}

  def load_state_dict(self, state_dict):
    self.current_step = state_dict["current_step"]

In [69]:
def load_training_data(max_samples: int = 10000):
  dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
  return [item["text"] for item in dataset.take(max_samples)]

def create_optimizer(model, config):
  to_decay = []
  no_decay = []
  for name, param in model.named_parameters():
    if not param.requires_grad:
      continue
    if param.dim() <= 1 or "norm" in name.lower() or "bias" in name.lower():
      no_decay.append(param)
    else:
      to_decay.append(param)

  adamw = torch.optim.AdamW([
      {"params": to_decay, "weight_decay": config.weight_decay},
      {"params": no_decay, "weight_decay": 0.0}
  ], lr = config.max_lr, betas = (config.beta1, config.beta2), eps = config.eps)

  return adamw

In [70]:
def train(model, train_dataset, config, device: None, save_dir = "checkpoints"):
  os.makedirs(save_dir, exist_ok = True)
  model = model.to(device)
  model.train()

  dataloader = torch.utils.data.DataLoader(train_dataset, batch_size = config.batch_size, shuffle = True, drop_last = True)

  optimizer = create_optimizer(model, config)

  scheduler = CosineWarmupScheduler(optimizer, warmup_steps=config.warmup_steps,
                                    max_steps = config.max_steps, max_lr =config.max_lr,
                                    min_lr = config.min_lr)
  step_count = 0
  total_loss = 0
  loss_hist = []

  scaler = torch.cuda.amp.GradScaler()
  while step_count < config.max_steps:
    for input_ids, target_ids in dataloader:
      input_ids = input_ids.to(device)
      target_ids = target_ids.to(device)

      optimizer.zero_grad()

      with torch.cuda.amp.autocast():
          logits, loss = model.forward(input_ids, target_ids)
      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.)
      scaler.step(optimizer)
      scaler.update()

      total_loss += loss.item()
      loss_hist.append((step_count, loss.item()))
      if step_count % 500 == 0:
        print(f"Step count: {step_count} | Loss: {loss.item()}")

      step_count += 1
  return loss_hist


def plot_loss(loss_hist):
  plt.figure(figsize=(10, 5))
  steps, losses = zip(*loss_hist)
  plt.plot(steps, losses)
  plt.xlabel("Step")
  plt.ylabel("Loss")
  plt.title("Training Loss")

In [74]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
  # Params
  batch_size : int = 2
  vocab_size : int = 50257
  embeddings_dim : int = 768
  head_count : int = 6
  transformer_count : int = 6
  max_seq_len : int = 512

  # Regularization
  embd_dropout : int = 0.1
  dropout : int = 0.1
  layer_norm_epsilon : int = 1e-5

  # Training
  weight_decay : int = 0.1
  max_lr : int = 1e-4
  min_lr: float = 1e-5
  warmup_steps : int  = 2000
  max_steps : int = 10000
  grad_accum_steps : int = 4
  beta1 : int = 0.9
  beta2 : int = 0.95
  eps : int = 1e-8

In [72]:
class GPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config

    self.embeddings = Embedding(config.vocab_size, config.embeddings_dim)

    self.embd_dropout = nn.Dropout(config.embd_dropout)

    self.layers = nn.ModuleList([TransformerBlock(config.embeddings_dim,
                                                  config.head_count,
                                                  config.dropout) for _ in range(config.transformer_count)])

    self.RMSNorm = RMSNorm(config.embeddings_dim)

    self.lm_head = nn.Linear(config.embeddings_dim, config.vocab_size, bias = False)

    # weight tying
    self.embeddings.embed.weight = self.lm_head.weight

    self.apply(self._init_weights)
    print(f"GPT initialized with {self._count_params():,} parameters")

  # helper
  def _init_weights(self, module: nn.Module):
      if isinstance(module, nn.Linear):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
          if module.bias is not None:
              torch.nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  # for fun
  def _count_params(self):
    return sum(p.numel() for p in self.parameters())

  def make_mask(self, seq_len) -> torch.Tensor:
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

  def forward(self, input_ids: torch.Tensor, targets: torch.Tensor = None) -> tuple:

    batch_size, seq_len = input_ids.shape

    x = self.embeddings(input_ids)
    x = self.embd_dropout(x)

    mask = self.make_mask(seq_len)

    for layer in self.layers:
      x = layer(x, mask)

    x = self.RMSNorm(x)
    logits = self.lm_head(x)

    loss = None

    if targets is not None:
      reshaped_logits = logits.view(-1, logits.shape[-1])
      reshaped_targets = targets.view(-1)
      loss = F.cross_entropy(reshaped_logits, reshaped_targets)

    return (logits, loss)

  @torch.no_grad()
  def generate(self, input_ids: torch.Tensor, max_new_tokens: int, temperature: float = 1, top_k : int = None, top_p: float = None):

    self.eval()

    for _ in range(max_new_tokens):
      if input_ids.shape[1] > self.config.max_seq_len:
        input_ids = input_ids[:, -self.config.max_seq_len:]

      logits, _ = self(input_ids)

      last = logits[:, -1, :]

      last = last / temperature

      if top_k is not None:
        v, _ = torch.topk(last, top_k)
        last[last < v[:, [-1]]] = float('-inf')

      probs = F.softmax(last, dim = -1)
      next_token = torch.multinomial(probs, num_samples = 1)

      input_ids = torch.cat((input_ids, next_token), dim = 1)

    return input_ids


In [ ]:
from typing_extensions import Text
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

config = GPTConfig()
texts = load_training_data(max_samples=10000)
tokenizer = Tokenizer()
dataset = TextDataset(texts, tokenizer)
model = GPT(config)

loss_hist = train(model, dataset, config, device)

Using device: cuda


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Total tokens: 10279481
GPT initialized with 95,230,464 parameters


/tmp/ipykernel_44659/807123052.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_44659/807123052.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Step count: 0 | Loss: 10.93266487121582
Step count: 500 | Loss: 6.556349277496338
Step count: 1000 | Loss: 6.325711727142334
Step count: 1500 | Loss: 6.0733489990234375
Step count: 2000 | Loss: 5.867352485656738
Step count: 2500 | Loss: 5.809772968292236


In [ ]:
input_ids = tokenizer.encode(" ")
input_ids = torch.tensor(input_ids).unsqueeze(0).to(device)
output = model.generate(input_ids, max_new_tokens=30)
print(tokenizer.decode(output[0].tolist()))